
# Hippo cNMF program-effect analyses

This notebook is written in **Jupyter style**. It uses the cNMF spreadsheet only.

The core vector used throughout is:

\[
\text{target vector} = [\text{approx\_log2FC for each day-program feature}]
\]

Rows are target genes. Columns are day-program features such as `D1 P13`, `D1 P30`, `D2 P15`, `D2 P17`, `D3 P34`, etc.

Main analyses:
1. Build target × day-program effect matrix.
2. Compute biologically oriented D1/D2/D3 fate-bias scores.
3. Embed targets in unbiased effect space.
4. Compute Hippo centroid similarity.
5. Test Hippo compactness using a strength-matched null.
6. Make signed significant program-effect overlap matrices.


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.manifold import MDS
from scipy.stats import hypergeom

# Edit this if your file is elsewhere
XLSX = "cNMF_50_0_2 032526_6co.xlsx"

OUTDIR = "hippo_cnmf_notebook_outputs"
os.makedirs(OUTDIR, exist_ok=True)

EFFECT_COL = "approx_log2FC"
PVAL_COL = "adj_pval"

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})



## 1. Load cNMF perturbation-effect table


In [ ]:

pert = pd.read_excel(XLSX, sheet_name="Perturbation Association 1")

pert["program_name"] = pd.to_numeric(pert["program_name"], errors="coerce")
pert = pert[pert["program_name"].notna()].copy()
pert["program_name"] = pert["program_name"].astype(int)

pert[EFFECT_COL] = pd.to_numeric(pert[EFFECT_COL], errors="coerce")
if PVAL_COL in pert.columns:
    pert[PVAL_COL] = pd.to_numeric(pert[PVAL_COL], errors="coerce")

pert["feature"] = pert["Sample"].astype(str) + " P" + pert["program_name"].astype(str)

print(pert.shape)
pert.head()



## 2. Build the main target × day-program effect vector

This is the key vector:

```text
v_target = [effect on D0 P1, effect on D0 P2, ..., effect on D3 P50]
```

Then we make a row-z-scored version to compare **patterns** of effects across programs rather than only magnitude.


In [ ]:

effect_mat = pert.pivot_table(
    index="target_name",
    columns="feature",
    values=EFFECT_COL,
    aggfunc="mean"
).sort_index()

# Fill occasional missing values by feature mean.
X = effect_mat.apply(lambda col: col.fillna(col.mean()), axis=0)

# Row-z-score each target profile.
Xz = (
    X.sub(X.mean(axis=1), axis=0)
     .div(X.std(axis=1).replace(0, np.nan), axis=0)
     .fillna(0)
)

# Raw strength used for strength-matched nulls.
target_strength = np.sqrt((X ** 2).sum(axis=1))

print("Targets:", X.shape[0])
print("Day-program features:", X.shape[1])
X.head()



## 3. Define Hippo set and fate-bias scores

Positive score orientation below is **alternative / failure bias**:

```text
D1 alternative bias = P13 - P30
D2 alternative bias = P17 - P15
D2 YAP/Lat-Meso bias = mean(P17,P34) - mean(P15,P23)
D3 alternative endpoint bias = mean(P27,P34,P35) - mean(P2,P3,P8,P11)
```


In [ ]:

def get_effect(day, program):
    col = f"{day} P{program}"
    if col not in X.columns:
        raise KeyError(f"Missing {col}")
    return X[col]

hippo_candidates = [
    "NF2", "LATS1", "LATS2", "SAV1", "STK3", "STK4",
    "MOB1A", "MOB1B", "TAOK1", "TAOK2", "TAOK3",
    "AMOT", "AMOTL1", "AMOTL2"
]
hippo = [g for g in hippo_candidates if g in X.index]
print("Hippo genes present:", hippo)

scores = pd.DataFrame(index=X.index)

scores["D1_alt_P13_minus_P30"] = get_effect("D1", 13) - get_effect("D1", 30)
scores["D2_alt_P17_minus_P15"] = get_effect("D2", 17) - get_effect("D2", 15)
scores["D2_YAPLat_minus_PreEC"] = (
    (get_effect("D2", 17) + get_effect("D2", 34)) / 2
    - (get_effect("D2", 15) + get_effect("D2", 23)) / 2
)
scores["D3_ALT_minus_EC"] = (
    (get_effect("D3", 27) + get_effect("D3", 34) + get_effect("D3", 35)) / 3
    - (get_effect("D3", 2) + get_effect("D3", 3) + get_effect("D3", 8) + get_effect("D3", 11)) / 4
)

scores["is_Hippo_core"] = scores.index.isin(hippo)
scores.reset_index().rename(columns={"target_name": "target"}).to_csv(
    os.path.join(OUTDIR, "target_fate_bias_scores.csv"),
    index=False
)

scores.loc[hippo]



## 4. Helper plotting functions


In [ ]:

def finish_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def savefig(fig, name):
    fig.savefig(os.path.join(OUTDIR, f"{name}.png"), dpi=350, bbox_inches="tight")
    fig.savefig(os.path.join(OUTDIR, f"{name}.pdf"), bbox_inches="tight")
    plt.show()

def scatter_with_hippo(x, y, xlabel, ylabel, title, name):
    d = scores[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    ax.scatter(d[x], d[y], s=22, alpha=0.50, linewidths=0, label="Other targets")

    h = [g for g in hippo if g in d.index]
    ax.scatter(d.loc[h, x], d.loc[h, y], s=95, marker="*", alpha=0.95, label="Hippo core")

    for g in h:
        ax.annotate(
            g, (d.loc[g, x], d.loc[g, y]),
            xytext=(5, 5), textcoords="offset points",
            fontsize=8.5, fontweight="bold"
        )

    ax.axhline(0, linestyle="--", linewidth=0.8)
    ax.axvline(0, linestyle="--", linewidth=0.8)

    r, p = stats.pearsonr(d[x], d[y])
    ptxt = f"{p:.1e}" if p < 1e-3 else f"{p:.3f}"
    m, b = np.polyfit(d[x], d[y], 1)
    xx = np.linspace(d[x].min(), d[x].max(), 200)
    ax.plot(xx, m * xx + b, linewidth=1.1)

    ax.text(
        0.04, 0.96,
        f"r = {r:.2f}\np = {ptxt}\nn = {len(d)}",
        transform=ax.transAxes,
        ha="left", va="top", fontsize=9
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    finish_axes(ax)
    leg = ax.legend(loc="best", frameon=True, fontsize=9)
    leg.get_frame().set_linewidth(0.5)
    fig.tight_layout()
    savefig(fig, name)



## 5. D1 → D2 → D3 fate-bias scatter plots


In [ ]:

scatter_with_hippo(
    "D1_alt_P13_minus_P30",
    "D2_alt_P17_minus_P15",
    "D1 alternative bias: P13 - P30",
    "D2 alternative bias: P17 - P15",
    "D1 Hippo-associated shift aligns with the D2 fate split",
    "01_D1_alt_vs_D2_alt_Hippo"
)

scatter_with_hippo(
    "D2_alt_P17_minus_P15",
    "D3_ALT_minus_EC",
    "D2 alternative bias: P17 - P15",
    "D3 alternative endpoint bias: ALT - EC",
    "D2 fate split predicts D3 endpoint bias",
    "02_D2_alt_vs_D3_alt_Hippo"
)

scatter_with_hippo(
    "D1_alt_P13_minus_P30",
    "D3_ALT_minus_EC",
    "D1 alternative bias: P13 - P30",
    "D3 alternative endpoint bias: ALT - EC",
    "D1 shift foreshadows D3 endpoint bias",
    "03_D1_alt_vs_D3_alt_Hippo"
)



## 6. Hippo trajectory across fate-bias axes

This is the cNMF analogue of an asynchronous-development trajectory: Hippo perturbations should sit high on D1 alternative bias, D2 alternative bias, and D3 endpoint bias.


In [ ]:

traj_cols = ["D1_alt_P13_minus_P30", "D2_alt_P17_minus_P15", "D3_ALT_minus_EC"]
traj_labels = ["D1\nP13-P30", "D2\nP17-P15", "D3\nALT-EC"]

bg = scores[traj_cols].dropna()
x = np.arange(len(traj_cols))

fig, ax = plt.subplots(figsize=(5.8, 4.8))

for i, col in enumerate(traj_cols):
    ax.boxplot(
        bg[col].values,
        positions=[i],
        widths=0.28,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(facecolor="0.9", edgecolor="0.5", linewidth=0.8),
        medianprops=dict(color="0.25", linewidth=1.0),
        whiskerprops=dict(color="0.5", linewidth=0.8),
        capprops=dict(color="0.5", linewidth=0.8),
    )

for g in hippo:
    vals = scores.loc[g, traj_cols].values.astype(float)
    ax.plot(x, vals, marker="o", linewidth=1.4, alpha=0.9)
    ax.text(x[-1] + 0.05, vals[-1], g, fontsize=8.5, fontweight="bold", va="center")

ax.axhline(0, linestyle="--", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(traj_labels)
ax.set_ylabel("Alternative/failure-bias effect")
ax.set_title("Hippo-core perturbations show early-to-late fate bias", fontweight="bold")
ax.set_xlim(-0.45, 2.55)
finish_axes(ax)
fig.tight_layout()
savefig(fig, "04_Hippo_alt_bias_trajectory_with_background")



## 7. Unbiased target effect-space MDS

Vector used here:

```text
Xz[target, :] = row-z-scored full day-program effect profile
```

Distance between targets:

```text
distance = sqrt(2 * (1 - Pearson correlation))
```


In [ ]:

# Target-target correlation of full row-z-scored profiles
corr = np.corrcoef(Xz.values)
corr = np.nan_to_num(corr, nan=0.0)
dist = np.sqrt(np.maximum(0, 2 * (1 - corr)))

mds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    random_state=1,
    normalized_stress="auto"
)
emb = mds.fit_transform(dist)

mds_df = pd.DataFrame(
    emb,
    index=Xz.index,
    columns=["MDS1", "MDS2"]
)
mds_df["is_Hippo_core"] = mds_df.index.isin(hippo)
mds_df["strength"] = target_strength.loc[mds_df.index]
mds_df.reset_index().rename(columns={"target_name": "target"}).to_csv(
    os.path.join(OUTDIR, "target_effect_space_mds_embedding.csv"),
    index=False
)

fig, ax = plt.subplots(figsize=(6.0, 5.2))
other = mds_df[~mds_df["is_Hippo_core"]]
h = mds_df[mds_df["is_Hippo_core"]]

ax.scatter(other["MDS1"], other["MDS2"], s=22, alpha=0.45, linewidths=0)
ax.scatter(h["MDS1"], h["MDS2"], s=110, marker="*", alpha=0.95, label="Hippo core")

for g, row in h.iterrows():
    ax.annotate(g, (row["MDS1"], row["MDS2"]), xytext=(5, 5), textcoords="offset points",
                fontsize=8.5, fontweight="bold")

ax.set_xlabel("MDS1")
ax.set_ylabel("MDS2")
ax.set_title("Unbiased target effect-space embedding", fontweight="bold")
finish_axes(ax)
ax.legend(frameon=True, fontsize=9)
fig.tight_layout()
savefig(fig, "05_unbiased_target_effect_space_MDS_Hippo")



## 8. Hippo centroid similarity

The Hippo centroid is the mean row-z-scored profile of the Hippo-core genes:

```text
Hippo_centroid = mean(Xz[NF2], Xz[LATS2], Xz[SAV1], Xz[TAOK1], Xz[AMOTL2])
```

For every target:

```text
similarity = Pearson correlation(target profile, Hippo centroid)
```


In [ ]:

hippo_centroid = Xz.loc[hippo].mean(axis=0)

similarity = Xz.apply(lambda row: stats.pearsonr(row.values, hippo_centroid.values)[0], axis=1)
sim_df = pd.DataFrame({
    "target": similarity.index,
    "similarity_to_Hippo_centroid": similarity.values,
    "is_Hippo_core": similarity.index.isin(hippo),
    "strength": target_strength.loc[similarity.index].values,
}).sort_values("similarity_to_Hippo_centroid", ascending=False)

sim_df.to_csv(os.path.join(OUTDIR, "target_similarity_to_hippo_centroid.csv"), index=False)
sim_df.head(20)


In [ ]:

topn = 30
plot_df = sim_df.head(topn).iloc[::-1].copy()

fig, ax = plt.subplots(figsize=(6.0, 7.0))
colors = ["tab:orange" if x else "0.55" for x in plot_df["is_Hippo_core"]]
ax.barh(plot_df["target"], plot_df["similarity_to_Hippo_centroid"], color=colors)
ax.axvline(0, linestyle="--", linewidth=0.8)
ax.set_xlabel("Correlation to Hippo centroid")
ax.set_title(f"Top {topn} targets nearest the Hippo effect-profile centroid", fontweight="bold")
finish_axes(ax)
fig.tight_layout()
savefig(fig, "06_targets_nearest_Hippo_centroid")



## 9. Hippo compactness with strength-matched null

Observed statistic:

```text
mean pairwise correlation among Hippo-core full effect profiles
```

Null:

```text
random target sets of same size, matched by target-effect strength bins
```


In [ ]:

def mean_pairwise_corr(targets, corr_df):
    targets = list(targets)
    vals = []
    for i in range(len(targets)):
        for j in range(i + 1, len(targets)):
            vals.append(corr_df.loc[targets[i], targets[j]])
    return float(np.mean(vals))

corr_df = pd.DataFrame(corr, index=Xz.index, columns=Xz.index)

obs_compactness = mean_pairwise_corr(hippo, corr_df)

strength_df = pd.DataFrame({
    "target": target_strength.index,
    "strength": target_strength.values
}).set_index("target")
strength_df["bin"] = pd.qcut(strength_df["strength"], q=10, labels=False, duplicates="drop")

rng = np.random.default_rng(11)
n_perm = 10000
null = np.zeros(n_perm)

hippo_bins = strength_df.loc[hippo, "bin"]

for k in range(n_perm):
    sampled = []
    for g, b in hippo_bins.items():
        pool = strength_df[
            (strength_df["bin"] == b) & (~strength_df.index.isin(hippo))
        ].index.to_numpy()
        if len(pool) == 0:
            pool = strength_df[~strength_df.index.isin(hippo)].index.to_numpy()
        sampled.append(rng.choice(pool))
    null[k] = mean_pairwise_corr(sampled, corr_df)

p_emp = (np.sum(null >= obs_compactness) + 1) / (n_perm + 1)
z = (obs_compactness - null.mean()) / null.std(ddof=1)

compact_df = pd.DataFrame([{
    "hippo_genes": ",".join(hippo),
    "n_hippo": len(hippo),
    "observed_mean_pairwise_corr": obs_compactness,
    "null_mean": null.mean(),
    "null_sd": null.std(ddof=1),
    "z_score": z,
    "empirical_p_greater": p_emp,
    "n_permutations": n_perm,
}])
compact_df.to_csv(os.path.join(OUTDIR, "hippo_profile_compactness_strength_matched_null.csv"), index=False)
compact_df


In [ ]:

fig, ax = plt.subplots(figsize=(5.5, 4.6))
ax.hist(null, bins=45, alpha=0.75)
ax.axvline(obs_compactness, linewidth=2.0, label=f"Hippo = {obs_compactness:.2f}")
ax.axvline(null.mean(), linestyle="--", linewidth=1.0, label="Null mean")
ax.set_xlabel("Mean pairwise profile correlation")
ax.set_ylabel("Strength-matched random sets")
ax.set_title("Hippo-core effect-profile compactness", fontweight="bold")
ax.text(0.04, 0.96, f"z = {z:.2f}\np = {p_emp:.4f}", transform=ax.transAxes,
        ha="left", va="top", fontsize=9.5)
finish_axes(ax)
ax.legend(frameon=True, fontsize=9)
fig.tight_layout()
savefig(fig, "07_hippo_profile_compactness_strength_matched_null")



## 10. Signed significant target-program feature sets

For each target, define a set:

```text
(feature, UP)   if adj_pval < 0.05 and approx_log2FC >= 0.25
(feature, DOWN) if adj_pval < 0.05 and approx_log2FC <= -0.25
```

This is analogous to DEG-set overlap in the Nature paper, except the “features” are signed cNMF program effects.


In [ ]:

sig = pert.copy()
sig = sig[
    sig[PVAL_COL].notna()
    & (sig[PVAL_COL] < 0.05)
    & (sig[EFFECT_COL].abs() >= 0.25)
].copy()

sig["direction"] = np.where(sig[EFFECT_COL] > 0, "UP", "DOWN")
sig["signed_feature"] = sig["feature"] + ":" + sig["direction"]

target_sets = sig.groupby("target_name")["signed_feature"].apply(set).to_dict()

# Use Hippo + nearest targets for an interpretable overlap matrix
targets_for_overlap = []
targets_for_overlap.extend(hippo)
targets_for_overlap.extend([
    t for t in sim_df["target"].head(20).tolist()
    if t not in targets_for_overlap
])
targets_for_overlap = [t for t in targets_for_overlap if t in target_sets]

M = len(pert["feature"].unique()) * 2  # UP and DOWN features

overlap = pd.DataFrame(index=targets_for_overlap, columns=targets_for_overlap, dtype=float)
pvals = pd.DataFrame(index=targets_for_overlap, columns=targets_for_overlap, dtype=float)

for a in targets_for_overlap:
    A = target_sets.get(a, set())
    for b in targets_for_overlap:
        B = target_sets.get(b, set())
        k = len(A & B)
        overlap.loc[a, b] = k
        # Hypergeometric survival: P(X >= k)
        pval = hypergeom.sf(k - 1, M, len(A), len(B)) if k > 0 else 1.0
        pvals.loc[a, b] = pval

overlap.to_csv(os.path.join(OUTDIR, "signed_program_feature_overlap_counts.csv"))
pvals.to_csv(os.path.join(OUTDIR, "signed_program_feature_overlap_pvalues.csv"))

overlap.head()


In [ ]:

fig, ax = plt.subplots(figsize=(7.0, 6.5))
im = ax.imshow(overlap.values, aspect="auto")
ax.set_xticks(range(len(overlap.columns)))
ax.set_xticklabels(overlap.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(overlap.index)))
ax.set_yticklabels(overlap.index, fontsize=8)

for i in range(overlap.shape[0]):
    for j in range(overlap.shape[1]):
        val = overlap.iloc[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{int(val)}", ha="center", va="center", fontsize=6)

ax.set_title("Overlap of signed significant cNMF program effects", fontweight="bold")
cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
cbar.set_label("Shared signed features")
fig.tight_layout()
savefig(fig, "08_signed_program_feature_overlap_matrix")



## 11. Unsupervised target-program heatmap

Rows: Hippo genes plus strongest target perturbations.  
Columns: most variable day-program features among selected rows.


In [ ]:

top_strength = target_strength.sort_values(ascending=False).head(100).index.tolist()
rows = []
for g in hippo + top_strength:
    if g in Xz.index and g not in rows:
        rows.append(g)

heat = Xz.loc[rows].copy()
top_features = heat.var(axis=0).sort_values(ascending=False).head(60).index
heat = heat[top_features]

# Cluster rows by correlation distance
row_dist = 1 - np.corrcoef(heat.values)
row_dist = np.nan_to_num(row_dist, nan=1.0)
Z = linkage(squareform(row_dist, checks=False), method="average")
order = leaves_list(Z)
heat_ord = heat.iloc[order]

fig, ax = plt.subplots(figsize=(12.0, max(7.0, 0.08 * heat_ord.shape[0])))
im = ax.imshow(heat_ord.values, aspect="auto", vmin=-2.5, vmax=2.5, cmap="RdBu_r")

ax.set_xticks(range(len(heat_ord.columns)))
ax.set_xticklabels(heat_ord.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(heat_ord.index)))
ax.set_yticklabels(heat_ord.index, fontsize=6)

# Bold-ish mark Hippo genes with asterisk in labels
for label in ax.get_yticklabels():
    if label.get_text() in hippo:
        label.set_fontweight("bold")
        label.set_color("tab:orange")

ax.set_title("Unsupervised target-program effect heatmap", fontweight="bold")
cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label("row-z-scored effect")
fig.tight_layout()
savefig(fig, "09_unsupervised_target_program_heatmap_top_strength")



## 12. Summary outputs


In [ ]:

print("Output directory:", OUTDIR)
print("Hippo genes:", hippo)
print("Compactness result:")
print(compact_df.to_string(index=False))
print("\nFiles:")
for fn in sorted(os.listdir(OUTDIR)):
    print(os.path.join(OUTDIR, fn))
